# Readable verification of `SAVED_CANDIDATES`

This notebook checks the **same saved certificates, using the same mathematics**
as `Verify_SAVED_CANDIDATES_Colab.ipynb` in the previous bundle. The numerical
interfaces and the reference/update decomposition are retained. Dense statements
have been expanded, and explanations have been placed beside the relevant code.

This is verification only: it does not optimize, find a new partition, or choose
a new R. It reads those from your existing `SAVED_CANDIDATES.py`.

**To run:** execute cells from top to bottom and upload that file when prompted.
The original file from the previous bundle can be used unchanged.

**To read:** begin with the mathematical outline below. The `Envelope` guide
explains the class, `__init__`, attributes, caching, and each method. You do not
need to understand Python classes before using the notebook.

The important promise is an inequality, not a guess at the maximum:

$$H(r)=\sum_jc_j\bigl(\kappa_d(s_jr)-s_j^{-d}\kappa_d(r/s_j)\bigr)-\kappa_d(r),$$
$$\mathcal I_d=\sup_{r\ge0}|H(r)|\le U.$$

For each saved interval [a,b], the verifier computes

$$B_{[a,b]}=\max\{\overline{|H(a)|},\overline{|H(b)|}\}
  +\frac{M_{2,[a,b]}(b-a)^2}{8},$$

where $M_{2,[a,b]}\ge\sup_{[a,b]}|H''|$. It checks that the saved interval bound
is at least this computed bound. It checks the tail separately, then checks the
saved global U dominates every saved component. Therefore it covers the whole
half-line, not just sampled points.


In [ ]:
%pip install -q python-flint==0.9.0


## Settings

`DIGITS` controls numerical ball-arithmetic precision. `PREC` is different: it
is the desired absolute width of the enclosure **after taking the dth root**.
Set `DIMENSIONS=[6]` for a quick initial run; `None` verifies every saved dimension.
Turn `RUN_CLASS_EXAMPLE` on to see a small example with a hand-checkable answer.


In [ ]:
from fractions import Fraction
from dataclasses import dataclass
from pathlib import Path
import importlib.util
import json
import math
import time
from flint import arb, fmpq, ctx

DIGITS = 180
PREC = "0.0001"
DIMENSIONS = None
CANDIDATE_FILE = "SAVED_CANDIDATES.py"
RUN_CLASS_EXAMPLE = False

ctx.dps = DIGITS


## 1. Exact inputs and safe output rounding

- `fmpq` represents exact rational numbers. Supply decimal/fraction **strings**,
  such as `"0.01"` or `"3/7"`; the input helper rejects Python floats.
- `arb` represents a midpoint and error radius. All core evaluations include
  numerical error. `.upper()` and `.lower()` give outward bounds.
- `.abs_upper()` bounds the absolute value from above. `.abs_lower()` bounds it
  from below; this is zero if the ball contains zero.
- The original directed float export is retained for displaying/serializing
  results and representing the exported lower root. `nextafter` moves that
  export in a conservative direction. It does not turn a sample into an upper
  bound for an interval; the derivative correction performs that job.

Useful Python notation: `**` means a power; `+=` means add to the current value;
`zip(A,B)` pairs corresponding entries of two lists; `map(f,A)` applies f to
each entry. `dictionary[key]` accesses a stored value.

| Python notation | Meaning in this code |
|---|---|
| `range(2, K+1)` | Integers 2 through K, including K |
| `n // 2` | Integer part of n/2 |
| `n % 2` | 0 for an even integer, 1 for an odd integer |
| `items.append(x)` | Add x at the end of a list |
| `c[:n]`, `c[n:]` | First n coefficients, then the remaining coefficients |
| `return x` | Give x back to the function's caller |
| `if key not in cache` | Only compute a value if it has not been saved yet |


In [ ]:
def rat(x):
    """Convert exact input to an exact rational number. Python floats are rejected.
Strings such as "0.01" and "3/7" are allowed."""
    if isinstance(x, float):
        raise TypeError('Use exact strings, not floats, in certificates.')
    v = Fraction(x) if isinstance(x, (str, int, Fraction)) else x
    return fmpq(v.numerator, v.denominator) if isinstance(v, Fraction) else fmpq(v)


def upper_float(x):
    """For display/export only: round a ball upper endpoint toward +infinity."""
    return math.nextafter(float(arb(x).upper()), math.inf)


def lower_float(x):
    """For display/export only: round a ball lower endpoint toward -infinity."""
    return math.nextafter(float(arb(x).lower()), -math.inf)


def upper_string(x):
    """Export a conservative upper bound as an exact rational string."""
    return str(Fraction.from_float(upper_float(x)))


def lower_string(x):
    """Export a conservative lower bound as an exact rational string."""
    return str(Fraction.from_float(lower_float(x)))


def root_gap(u, l, d):
    """Enclose the difference of the upper and lower dth roots, rounding outward."""
    return (arb(u).root(d).upper() - arb(l).root(d).lower()).upper()


## 2. The kernel and its tail bound

With $a=d/2$, $\nu=a-1$, the two kernel formulas are equivalent:

$$\kappa_d(r)=\Gamma(a)(\pi r)^{-\nu}J_\nu(2\pi r)
={}_0F_1(;a;-\pi^2r^2).$$

The hypergeometric form avoids division by zero when the ball contains zero.
In `(-z*z).hypgeom_0f1(a)`, the ball `-z*z` is the argument and `a` is the
parameter of ${}_0F_1$.

`decay(r,d)` bounds $|\kappa_d(t)|$ for **every** $t\ge r>0$:

$$D_d(r)=\min\left\{1,
 \frac{0.8\,\Gamma(d/2)}{(\pi r)^{d/2-1}(2\pi r)^{1/3}}\right\}.$$

This uses the analytic inequality $|J_\nu(t)|\le0.8t^{-1/3}$ for
$t>0$, $\nu\ge0$, together with $|\kappa_d|\le1$. Since the resulting
power bound decreases with its argument, its value at r bounds every t>=r.
The same decay inequality and constant as in the original verifier are used.
The general H-tail is obtained by the triangle inequality with the scaled
arguments $s_jR$ and $R/s_j$.


In [ ]:
def kernel(r, d):
    """Enclose kappa_d(r). The hypergeometric branch is safe at zero.
All constants and special-function evaluations here use Arb balls."""
    r = arb(r)
    z = arb.pi() * r
    a = arb(d) / 2
    if r.contains(0):
        return (-z * z).hypgeom_0f1(a)
    return a.gamma() * (2 * z).bessel_j(a - 1) / z ** (a - 1)


def decay(r, d):
    """Bound |kappa_d(t)| for EVERY t >= r > 0, using the same decay inequality."""
    r = arb(r)
    a = arb(d) / 2
    z = arb.pi() * r
    return min(arb(1), (arb('0.8') * a.gamma() / z ** (a - 1) / (2 * z).root(3)).upper())


## 3. Understanding `class Envelope`

A class packages information and functions together. Here an `Envelope` object
stores ONE reference candidate, its reusable derivative coefficients, and caches.
It does not optimize or choose the subdivision.

When you write

```python
env = Envelope(d, c, s, order=12)
```

Python automatically calls `Envelope.__init__`. `self` inside a method means
that particular object, `env`. Thus `self.d = d` stores a value that can later
be read as `env.d`. Calling `env.value(r)` supplies `env` as `self` automatically.
`order=12` is a Taylor derivative order, **not** a number of modes or intervals.

### Methods: what each call returns

| Call | Purpose |
|---|---|
| `Envelope(d,c,s,order)` | Create an object and run `__init__` once |
| `env.value(r)` | Enclose the reference function at one point |
| `env.tail(R)` | Bound the entire reference tail, not only its endpoint |
| `env.kjet(t)` | Kernel value and derivatives 2 through K |
| `env.jet(r)` | Reference function value and derivatives 2 through K |
| `env.curvature(a,b)` | Bound the reference second derivative throughout [a,b] |

In this verifier, `value`, `jet`, and interval endpoints receive exact rational
points. For example, use `rat("0.1")`. The Taylor bound is what covers the
INTERIOR of an interval; point evaluations by themselves do not do that.

### The attributes you will see

| Attribute | Meaning |
|---|---|
| `d`, `c`, `s` | Dimension, exact coefficients, and positive scales |
| `K` | Highest derivative computed explicitly |
| `a`, `pi` | Arb constants $d/2$ and $\pi$ |
| `cb`, `sb` | Ball enclosures of the rational coefficients/scales |
| `terms[n]` | Coefficients for the nth Bessel derivative identity below |
| `forward[j][n]` | Chain-rule factor $c_js_j^n$ |
| `reverse[j][n]` | Chain-rule factor $c_js_j^{-d-n}$ |
| `remainder` | A global bound for the derivative of order $K+1$ |
| `M2` | The simpler global bound for the second derivative |
| `points` | Dictionary of already evaluated function values |
| `jets` | Dictionary of already evaluated derivative lists |

A cache changes runtime, not the mathematics. The keys are the strings of
exact rational points. Previously computed balls are reused.

### What `__init__` prepares

It stores the input, prepares derivative coefficients and chain-rule weights,
computes two global derivative bounds, and creates empty caches. These are
reused for all intervals, rather than recomputed for every evaluation.
If you change d, c, s or order, create a new Envelope object so that its
prepared weights and caches correspond to the new inputs.

For $\omega$ uniformly distributed on the unit sphere,

$$\sup_r|\kappa_d^{(n)}(r)|\le(2\pi)^n\mathbb E|\omega_1|^n,$$
$$\mathbb E|\omega_1|^n=
\frac{\Gamma(d/2)\Gamma((n+1)/2)}{\sqrt\pi\,\Gamma((d+n)/2)}.$$

Together with the chain rule, this gives

$$A_n=(2\pi)^n\mathbb E|\omega_1|^n
 \left(1+\sum_j|c_j|\bigl(s_j^n+s_j^{-d-n}\bigr)\right).$$

`self.remainder` stores an upper bound for $A_{K+1}$. It is **not yet the Taylor
error**. The error becomes $A_{K+1}\rho^{K-1}/(K-1)!$ when used in `curvature`.
The simpler bound is

$$M_2=\frac{4\pi^2}{d}
 \left(1+\sum_j|c_j|\bigl(s_j^2+s_j^{-d-2}\bigr)\right).$$

### What `kjet` computes

A "jet" here just means a collection of derivatives evaluated at one point.
For $a=d/2$ the identity used by the code is

$$\kappa_d^{(n)}(t)=\sum_{j=0}^{\lfloor n/2\rfloor}
\frac{n!\,2^{n-2j}(-\pi^2)^{n-j}}{j!(n-2j)!(a)_{n-j}}
 t^{n-2j}\kappa_{d+2(n-j)}(t).$$

`terms[n]` stores `(n-j, n-2*j, coefficient)` from this sum.
`a.rising(m)` is the rising factorial $(a)_m$.

For non-small arguments it reuses the Bessel recurrence

$$\kappa_{d+2(q+1)}(t)=
\frac{(a+q)(a+q-1)}{(\pi t)^2}
\left(\kappa_{d+2q}(t)-\kappa_{d+2(q-1)}(t)\right),\quad q\ge1.$$

For small arguments it evaluates the shifted kernels directly. All operations
are ball operations, including the recurrence. At zero, it uses the explicit
limits: odd derivatives vanish and

$$\kappa_d^{(2m)}(0)=\frac{(-\pi^2)^m(2m)!}{m!(a)_m}.$$

**Index 1 of the returned lists is an unused zero placeholder, not the first
derivative.** Indices 0 and 2,...,K carry the actual values used by the proof.
This original interface is kept so that the numerical workflow stays compatible.

### What `jet` computes

It applies the same formulas to H, using the stored chain-rule weights:

$$H^{(n)}(r)=-\kappa_d^{(n)}(r)+\sum_jc_j
 \left(s_j^n\kappa_d^{(n)}(s_jr)-s_j^{-d-n}\kappa_d^{(n)}(r/s_j)\right).$$

### What `curvature` computes

For midpoint $m=(a+b)/2$ and half-width $\rho=(b-a)/2$,

$$\sup_{[a,b]}|H''|\le
\sum_{n=2}^{K}\frac{|H^{(n)}(m)|\rho^{n-2}}{(n-2)!}
+\frac{A_{K+1}\rho^{K-1}}{(K-1)!}.$$

It keeps the smaller of this bound and the global M2. Thus higher derivatives
are used only to obtain a tighter bound for the SECOND derivative.


In [ ]:
class Envelope:
    """Prepare and evaluate one reference function H using Arb balls.

    H(r) = sum_j c[j] * (kappa_d(s[j]*r) - s[j]**(-d)*kappa_d(r/s[j]))
           - kappa_d(r).

    This class does not optimize coefficients or choose intervals. It stores
    quantities reused by the verifier. The original short attribute names are
    retained so that the numerical interfaces stay unchanged:

        d, c, s       dimension and exact rational coefficient/scale lists
        K             highest derivative order computed explicitly
        a             d/2 as an Arb number
        pi            an Arb enclosure of pi
        terms[n]      coefficient data for the nth derivative of kappa_d
        cb, sb        Arb versions of c and s
        forward[j][n] c[j] * s[j]**n
        reverse[j][n] c[j] * s[j]**(-d-n)
        remainder     global bound for the derivative |H^(K+1)|
        M2            global bound for |H''|
        points        cache of already evaluated H(r)
        jets          cache of already evaluated derivative lists

    IMPORTANT: remainder is not yet the Taylor error. curvature() multiplies
    it by half_width**(K-1)/(K-1)! to obtain the relevant remainder term.
    In derivative lists, index 1 is an unused placeholder, NOT H'(r).
    """

    def __init__(self, d, c, s, order=12):
        """Run once when Envelope(d,c,s,order) creates an object.

        1. Store exact input and reusable constants.
        2. Prepare coefficients for Bessel derivative identities.
        3. Prepare the chain-rule weights of the scaled kernels.
        4. Bound the highest needed derivative and the second derivative.
        5. Create empty caches to avoid repeated expensive evaluations.
        """
        # 1. Exact input. rat() rejects Python floats.
        self.d = int(d)
        self.c = list(map(rat, c))
        self.s = list(map(rat, s))
        self.K = order

        if d < 2 or len(c) != len(s) or any(scale <= 0 for scale in self.s):
            raise ValueError("Invalid candidate")
        if order < 2:
            raise ValueError("order must be >= 2")

        self.a = arb(d) / 2
        self.pi = arb.pi()

        # 2. Each term is stored as (dimension_shift, argument_power, coefficient).
        # For derivative order n and summation index j:
        #   dimension_shift = n-j, argument_power = n-2*j.
        # The full identity is written in the notebook immediately before this class.
        self.terms = {}
        for derivative_order in range(2, order + 1):
            self.terms[derivative_order] = []

            for pair_count in range(derivative_order // 2 + 1):
                dimension_shift = derivative_order - pair_count
                argument_power = derivative_order - 2 * pair_count

                numerator = (
                    arb(math.factorial(derivative_order) * 2**argument_power)
                    * (-self.pi**2)**dimension_shift
                )
                denominator = (
                    math.factorial(pair_count)
                    * math.factorial(argument_power)
                    * self.a.rising(dimension_shift)
                )
                coefficient = numerator / denominator

                self.terms[derivative_order].append(
                    (dimension_shift, argument_power, coefficient)
                )

        # Arb copies enclose the exact rational coefficients and scales.
        self.cb = list(map(arb, self.c))
        self.sb = list(map(arb, self.s))

        # 3. Differentiating kappa(s*r) n times supplies s**n.
        # Differentiating s**(-d)*kappa(r/s) supplies s**(-d-n).
        self.forward = []
        self.reverse = []
        for coefficient, scale in zip(self.cb, self.sb):
            positive_weights = []
            negative_weights = []
            for derivative_order in range(order + 1):
                positive_weights.append(coefficient * scale**derivative_order)
                negative_weights.append(coefficient * scale**(-d - derivative_order))
            self.forward.append(positive_weights)
            self.reverse.append(negative_weights)

        # 4a. Global bound on the derivative of order K+1.
        # moment = E|omega_1|**(K+1), for uniform omega on the unit sphere.
        derivative_order = order + 1
        moment = (
            self.a.gamma()
            * (arb(derivative_order + 1) / 2).gamma()
            / (
                self.pi.sqrt()
                * (arb(d + derivative_order) / 2).gamma()
            )
        )
        scaled_size = arb(0)
        for coefficient, scale in zip(self.cb, self.sb):
            scaled_size += abs(coefficient) * (
                scale**derivative_order + scale**(-d - derivative_order)
            )

        self.remainder = (
            (2 * self.pi)**derivative_order * moment * (1 + scaled_size)
        ).upper()

        # 4b. The simpler global bound |H''| <= (4*pi**2/d)*scaled_size.
        # The initial 1 accounts for the target term -kappa_d(r).
        scaled_size = arb(0)
        for coefficient, scale in zip(self.cb, self.sb):
            scaled_size += abs(coefficient) * (
                scale**2 + scale**(-d - 2)
            )
        self.M2 = (4 * self.pi**2 / d * (1 + scaled_size)).upper()

        # 5. Caches: dictionaries whose keys are exact rational point strings.
        self.points = {}
        self.jets = {}

    def tail(self, r):
        """Bound |H(t)| for EVERY t >= r, not only at the point r."""
        tail_upper = decay(r, self.d)
        for coefficient, scale in zip(self.cb, self.sb):
            tail_upper += abs(coefficient) * (
                decay(arb(r) * scale, self.d)
                + scale**(-self.d) * decay(arb(r) / scale, self.d)
            )
        return tail_upper.upper()

    def value(self, r):
        """Return an Arb enclosure of H(r), reusing a cached value if available."""
        key = str(r)
        if key not in self.points:
            argument = arb(r)
            value = -kernel(argument, self.d)

            for coefficient, scale in zip(self.cb, self.sb):
                value += coefficient * (
                    kernel(argument * scale, self.d)
                    - scale**(-self.d) * kernel(argument / scale, self.d)
                )

            if not value.is_finite():
                raise ArithmeticError("Nonfinite H; increase digits.")
            self.points[key] = value

        return self.points[key]

    def kjet(self, t):
        """Compute kappa_d(t) and derivatives 2,...,K at an Arb point t.

        Output[n] encloses kappa_d^(n)(t), except that output[1] is an UNUSED
        zero placeholder. It is kept only for compatibility with the old code.
        The shifted-dimension recurrence is a speed optimization; it does not
        change the Bessel derivative formula or omit rounding errors.
        """
        if t.is_zero():
            derivatives = [arb(1), arb(0)]
            for derivative_order in range(2, self.K + 1):
                if derivative_order % 2:
                    derivatives.append(arb(0))
                else:
                    half_order = derivative_order // 2
                    value = (
                        (-self.pi**2)**half_order
                        * math.factorial(derivative_order)
                        / (
                            math.factorial(half_order)
                            * self.a.rising(half_order)
                        )
                    )
                    derivatives.append(value)
            return derivatives

        # shifted_kernels[m] encloses kappa_(d+2*m)(t).
        if abs(t) < 1:
            # Direct evaluation avoids cancellation in the recurrence near zero.
            shifted_kernels = []
            for dimension_shift in range(self.K + 1):
                shifted_kernels.append(kernel(t, self.d + 2 * dimension_shift))
        else:
            shifted_kernels = [kernel(t, self.d), kernel(t, self.d + 2)]
            argument_squared = (self.pi * t)**2

            for dimension_shift in range(1, self.K):
                shifted_half_dimension = self.a + dimension_shift
                next_value = (
                    shifted_half_dimension * (shifted_half_dimension - 1)
                    / argument_squared
                    * (shifted_kernels[-1] - shifted_kernels[-2])
                )
                shifted_kernels.append(next_value)

        # powers[p] is t**p. Reuse multiplication rather than recomputing powers.
        powers = [arb(1)]
        for power in range(self.K):
            powers.append(powers[-1] * t)

        derivatives = [shifted_kernels[0], arb(0)]
        for derivative_order in range(2, self.K + 1):
            value = arb(0)
            for dimension_shift, argument_power, coefficient in self.terms[derivative_order]:
                value += (
                    coefficient * powers[argument_power]
                    * shifted_kernels[dimension_shift]
                )
            derivatives.append(value)

        return derivatives

    def jet(self, r):
        """Enclose H(r) and derivatives 2,...,K; index 1 is UNUSED.

        Start with derivatives of -kappa_d(r), then add derivatives of each
        scaled kernel using the precomputed chain-rule weights.
        """
        key = str(r)
        if key not in self.jets:
            argument = arb(r)
            derivatives = []
            for value in self.kjet(argument):
                derivatives.append(-value)

            used_orders = [0] + list(range(2, self.K + 1))
            for scale, forward_weights, reverse_weights in zip(
                self.sb, self.forward, self.reverse
            ):
                positive = self.kjet(argument * scale)
                negative = self.kjet(argument / scale)

                for derivative_order in used_orders:
                    derivatives[derivative_order] += (
                        forward_weights[derivative_order] * positive[derivative_order]
                        - reverse_weights[derivative_order] * negative[derivative_order]
                    )

            for derivative_order in used_orders:
                if not derivatives[derivative_order].is_finite():
                    raise ArithmeticError("Nonfinite derivative jet; increase digits.")

            self.jets[key] = derivatives
            self.points[key] = derivatives[0]

        return self.jets[key]

    def curvature(self, left, right):
        """Bound sup_[left,right] |H''| using Taylor's theorem.

        Let m=(left+right)/2 and rho=(right-left)/2. The local bound is
          sum_(n=2)^K |H^(n)(m)| rho**(n-2)/(n-2)!
          + remainder * rho**(K-1)/(K-1)!.
        Both this and self.M2 are valid; return the smaller upper bound.
        """
        midpoint = (left + right) / 2
        half_width = arb((right - left) / 2)
        derivatives = self.jet(midpoint)

        local_bound = arb(0)
        half_width_power = arb(1)
        for derivative_order in range(2, self.K + 1):
            local_bound += (
                derivatives[derivative_order].abs_upper()
                * half_width_power / math.factorial(derivative_order - 2)
            )
            half_width_power *= half_width

        taylor_remainder = (
            self.remainder * half_width_power / math.factorial(self.K - 1)
        )
        local_bound += taylor_remainder

        return min(self.M2, local_bound.upper())


### Optional hand-checkable class example

With `c=[1]` and `s=[1]`, each scaled-kernel pair cancels, so the reference
function is `-kappa_d`. In d=4 we know `H(0)=-1` and `H''(0)=pi**2`.
The example prints balls enclosing those values; it does not change your data.


In [ ]:
if RUN_CLASS_EXAMPLE:
    example = Envelope(4, ["1"], ["1"], order=12)
    print("H(0), expected -1:", example.value(fmpq(0)).str(20))
    print("H second derivative at 0, expected pi**2:", example.jet(fmpq(0))[2].str(20))
    print("Second derivative bound on [0,0.01]:",
          example.curvature(fmpq(0), rat("0.01")).str(20))


## 4. The saved reference/update split

`reference_count` splits the exported coefficients into two consecutive groups:

$$H_{\rm full}=H_{\rm reference}+H_{\rm update}.$$

The reference includes the target term $-\kappa_d(r)$; the update contains only
scaled-kernel pairs. It must NOT subtract a second copy of the target.
`Envelope` represents the reference group when verifying these saved candidates.
All exported coefficients still belong to the full function being certified.
This split lets the verifier retain cancellation in the large reference coefficients.

`Cell` below is only a record holding an interval's exact endpoints and its
reference curvature bound. Its `upper` field is unused during replay.
The `@dataclass` line asks Python to supply a simple constructor for this record;
`Cell(a,b,M2,None)` just stores those four values. It performs no analytic estimate.


In [ ]:
@dataclass
class Cell:
    """A small record for ONE interval.
left/right: exact endpoints. M2: reference curvature bound.
upper is unused during replay and is set to None; retained for compatibility."""
    left: object
    right: object
    M2: object
    upper: object


## 5. The second-derivative bounds and interpolation correction

`curvature_rest` bounds the variation of the reference second derivative around
the midpoint. It is the Taylor sum starting with derivative order 3, plus the
same remainder; it excludes the midpoint second derivative itself.

Let $V_{\rm ref}$ denote this variation bound and let
$A_{2,\rm update},A_{3,\rm update}$ be global derivative bounds for the update.
`verify_correction` computes two valid curvature bounds:

$$B_1=M_{2,\rm reference}+A_{2,\rm update},$$
$$B_2=|H_{\rm full}''(m)|+V_{\rm ref}+\rho A_{3,\rm update}.$$

The update derivative bounds have no initial 1, because the update contains
no target term. The code keeps the SIGNS of the midpoint contributions until
after they have been added; only then does it take an absolute upper bound.
This preserves valid cancellation.

Finally, the visible variables `endpoint_upper`, `curvature_bound`, and
`interpolation_error` implement exactly

$$U_{[a,b]}=\max\{\overline{|H(a)|},\overline{|H(b)|}\}
       +\min(B_1,B_2)\frac{(b-a)^2}{8}.$$

Midpoint evaluations also provide sampled lower bounds; they are not substituted
for the required upper bound over the entire interval. We differentiate H, not
its absolute value. The interpolation error and triangle inequality then bound
|H|, so zeros of H do not require differentiating a nonsmooth absolute value.


In [ ]:
def curvature_rest(env, left, right):
    """Bound the variation of the reference H'' around the midpoint.

    This is the Taylor sum starting at derivative order 3, plus its remainder.
    It EXCLUDES |H''(midpoint)|. The signed midpoint bound adds that separately.
    """
    half_width = arb((right - left) / 2)
    midpoint = (left + right) / 2
    derivatives = env.jet(midpoint)

    variation_bound = arb(0)
    half_width_power = half_width
    for derivative_order in range(3, env.K + 1):
        variation_bound += (
            derivatives[derivative_order].abs_upper()
            * half_width_power / math.factorial(derivative_order - 2)
        )
        half_width_power *= half_width

    taylor_remainder = (
        env.remainder * half_width_power / math.factorial(env.K - 1)
    )
    variation_bound += taylor_remainder
    return variation_bound.upper()


def verify_correction(env, cells, R, delta, ss):
    """Bound the FULL function: reference H plus the saved coefficient update.

    env represents H_reference. delta and ss describe H_update, which has NO
    extra -kappa term. Thus H_full = H_reference + H_update.

    For each interval, compute two valid bounds for |H_full''|, take their
    minimum, and add the interpolation correction M2*width**2/8. Finally
    combine all interval bounds with a separately computed infinite-tail bound.

    Returns (supremum_upper, sampled_lower, tail_upper, interval_upper_bounds).
    """
    dimension = env.d
    pi = arb.pi()

    # Exact zero terms contribute nothing and need no Bessel evaluations.
    active_terms = []
    for coefficient, scale in zip(delta, ss):
        if coefficient != 0:
            active_terms.append((arb(coefficient), arb(scale)))

    update_coefficients = []
    update_scales = []
    for coefficient, scale in active_terms:
        update_coefficients.append(coefficient)
        update_scales.append(scale)

    # A2_update >= sup |H_update''|.
    size = arb(0)
    for coefficient, scale in zip(update_coefficients, update_scales):
        size += abs(coefficient) * (scale**2 + scale**(-dimension - 2))
    update_second_bound = (4 * pi**2 / dimension * size).upper()

    # A3_update >= sup |H_update'''|, using E|omega_1|**3.
    third_moment = (
        (arb(dimension) / 2).gamma()
        / (arb.pi().sqrt() * (arb(dimension + 3) / 2).gamma())
    )
    size = arb(0)
    for coefficient, scale in zip(update_coefficients, update_scales):
        size += abs(coefficient) * (scale**3 + scale**(-dimension - 3))
    update_third_bound = ((2 * pi)**3 * third_moment * size).upper()

    full_values = {}
    sampled_lower = arb(0)

    def full_function_value(r):
        """Enclose H_full(r), remembering previously evaluated points."""
        key = str(r)
        if key not in full_values:
            argument = arb(r)
            value = env.value(r)
            for coefficient, scale in zip(update_coefficients, update_scales):
                value += coefficient * (
                    kernel(argument * scale, dimension)
                    - scale**(-dimension) * kernel(argument / scale, dimension)
                )
            if not value.is_finite():
                raise ArithmeticError("Nonfinite corrected endpoint")
            full_values[key] = value
        return full_values[key]

    def kernel_second(argument):
        """The same dimension-shift identity for kappa_d'' used earlier."""
        return (
            -4 * pi**2 / dimension * kernel(argument, dimension + 2)
            + 16 * pi**4 * argument**2 / (dimension * (dimension + 2))
            * kernel(argument, dimension + 4)
        )

    interval_upper_bounds = []
    for cell in cells:
        left = cell.left
        right = cell.right
        midpoint = (left + right) / 2
        width = arb(right - left)
        half_width = arb((right - left) / 2)

        left_value = full_function_value(left)
        right_value = full_function_value(right)
        sampled_lower = max(
            sampled_lower, left_value.abs_lower(), right_value.abs_lower()
        )
        sampled_lower = max(
            sampled_lower, full_function_value(midpoint).abs_lower()
        )

        # Keep the SIGNS here. Cancellation is used before taking abs_upper().
        argument = arb(midpoint)
        full_second_at_midpoint = env.jet(midpoint)[2]
        for coefficient, scale in zip(update_coefficients, update_scales):
            full_second_at_midpoint += coefficient * (
                scale**2 * kernel_second(argument * scale)
                - scale**(-dimension - 2) * kernel_second(argument / scale)
            )

        # Bound 1: reference curvature plus a global update curvature bound.
        first_curvature_bound = (cell.M2 + update_second_bound).upper()

        # Bound 2: signed full curvature at the midpoint, plus variation bounds.
        second_curvature_bound = (
            full_second_at_midpoint.abs_upper()
            + curvature_rest(env, left, right)
            + update_third_bound * half_width
        ).upper()

        curvature_bound = min(first_curvature_bound, second_curvature_bound)
        endpoint_upper = max(left_value.abs_upper(), right_value.abs_upper())

        # THIS is the correction for points between the endpoints.
        interpolation_error = curvature_bound * width**2 / 8
        interval_upper = (endpoint_upper + interpolation_error).upper()
        interval_upper_bounds.append(interval_upper)

    # The tail is bounded for EVERY r >= R, using the scaled arguments.
    tail_upper = env.tail(R)
    for coefficient, scale in zip(update_coefficients, update_scales):
        tail_upper += abs(coefficient) * (
            decay(arb(R) * scale, dimension)
            + scale**(-dimension) * decay(arb(R) / scale, dimension)
        )
    tail_upper = tail_upper.upper()

    supremum_upper = max(tail_upper, max(interval_upper_bounds))
    return supremum_upper, sampled_lower, tail_upper, interval_upper_bounds


## 6. Derived quantities and reading the data file

The input dictionary is loaded from your generated Python file. Only load your
own trusted file. No stored numerical bound is accepted merely because it was
read from disk.

Given a certified upper bound $U\ge\mathcal I_d$, monotonicity gives the lower bound

$$(4|B_1|U)^{-1/d}\le(4|B_1|\mathcal I_d)^{-1/d}.$$

The second reported lower bound divides this by $\sqrt d$.
The unit-ball volume used here is $|B_1|=\pi^{d/2}/\Gamma(d/2+1)$.


In [ ]:
def measures(d, upper, lower):
    """Convert certified supremum bounds to root and derived bounds for export.
These display floats are rounded in safe directions; they are not optimization data."""
    volume = arb.pi() ** (arb(d) / 2) / (arb(d) / 2 + 1).gamma()
    derived = 1 / (4 * volume * upper).root(d)
    return dict(I_upper_exact=upper_string(upper), root_upper=upper_float(upper.root(d)), root_lower=lower_float(lower.root(d)), root_gap_upper=upper_float(root_gap(upper, lower, d)), derived_lower=lower_float(derived), normalized_lower=lower_float(derived / arb(d).sqrt()))


def load_candidates(path):
    """Read the dictionary from the generated Python data file.
Use only a trusted SAVED_CANDIDATES.py file supplied or generated by you."""
    spec = importlib.util.spec_from_file_location('input_candidates', path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return getattr(m, 'SAVED_CANDIDATES', getattr(m, 'CANDIDATES', None))


## 7. Recompute first; compare with the saved bounds second

The calls below follow this order:

1. `verify_file` loops over selected dimensions.
2. `replay_entry` asks `verify_saved` to recompute the bounds.
3. `verify_saved` validates the partition and builds the reference and update.
4. `Envelope.curvature` and `verify_correction` compute interval/tail bounds.
5. `replay_entry` checks each stored bound, then the stored global U.

The partition must begin at 0, end exactly at the saved R, and increase strictly.
For every cell the checked chain is

$$\sup_{[a,b]}|H|\le U_{\rm computed,[a,b]}\le U_{\rm saved,[a,b]}.$$

The corresponding tail chain covers every $r\ge R$. The last check is

$$U_{\rm saved,global}\ge\max\{U_{\rm saved,tail},U_{\rm saved,all\ cells}\}.$$

If a ball comparison cannot establish the claimed inequality, verification
fails safely. It can be a precision issue rather than a counterexample; the
error message suggests increasing DIGITS or regenerating the saved certificate.
A successful check covers ALL intervals and the entire tail.

The precision flag is a separate check:

$$U_{\rm saved,global}^{1/d}-L_{\rm sampled}^{1/d}<\mathrm{PREC}.$$

The sampled lower bound comes from certified point values, not from the optimizer.
Certificate accuracy does not prove that the coefficients are globally optimal.


In [ ]:
def verify_saved(entry, d, digits=180):
    """Recompute a candidate certificate using its SAVED partition and R.
First reference_count terms form the reference; remaining terms form its update.
Stored interval/tail numerical bounds are not used in these calculations."""
    ctx.dps = digits
    n = entry['reference_count']
    c = entry['c']
    s = entry['s']
    if len(c) != len(s) or not isinstance(n, int) or (not 0 <= n <= len(c)):
        raise ValueError('Invalid coefficient/scales lengths or reference_count')
    if any((rat(x) <= 0 for x in s)):
        raise ValueError('All saved scales must be positive')
    env = Envelope(d, c[:n], s[:n], entry['taylor_order'])
    grid = list(map(rat, entry['partition']))
    R = rat(entry['R'])
    if grid[0] != 0 or grid[-1] != R or any((a >= b for a, b in zip(grid, grid[1:]))):
        raise ValueError('Partition does not exactly cover [0,R]')
    cells = [Cell(a, b, env.curvature(a, b), None) for a, b in zip(grid, grid[1:])]
    u, l, t, ubs = verify_correction(env, cells, R, list(map(rat, c[n:])), list(map(rat, s[n:])))
    return (measures(d, u, l), t, ubs)


def replay_entry(entry, d, digits=180, prec='0.0001'):
    """Check every recomputed bound against the corresponding stored upper bound.
Then check the global claim dominates ALL stored interval/tail bounds.
Only after these checks are the reported three quantities certified."""
    ctx.dps = digits
    result, tail, intervals = verify_saved(entry, d, digits)
    stored = entry['interval_upper_bounds']
    if len(stored) != len(intervals):
        raise ValueError('Wrong number of saved interval bounds')
    for i, (actual, claimed) in enumerate(zip(intervals, stored)):
        if not actual <= arb(rat(claimed)):
            raise ArithmeticError(f'Stored interval bound {i} not verified; increase digits or regenerate')
    if not tail <= arb(rat(entry['tail_upper'])):
        raise ArithmeticError('Stored tail bound not verified')
    claimed = arb(rat(entry['I_upper_exact']))
    components = [arb(rat(x)) for x in stored] + [arb(rat(entry['tail_upper']))]
    if any((x > claimed for x in components)):
        raise ValueError('Stored global bound is below a stored component')
    a = arb(d) / 2
    volume = arb.pi() ** a / (a + 1).gamma()
    derived = 1 / (4 * volume * claimed).root(d)
    lower = arb(repr(result['root_lower']))
    root_upper = claimed.root(d).upper()
    gap = (root_upper - lower).upper()
    return dict(d=d, verified=True, I_upper_exact=entry['I_upper_exact'], root_upper=upper_float(root_upper), root_lower=lower_float(lower), root_gap_upper=upper_float(gap), precision_achieved=gap < arb(rat(prec)), derived_lower=lower_float(derived), normalized_lower=lower_float(derived / arb(d).sqrt()), cells=len(intervals), R=entry['R'])


def verify_file(path, dimensions=None, digits=180, prec='0.0001', output='CERTIFIED_REPLAY.json'):
    """Run verification dimension by dimension, print results, and checkpoint JSON.
A failed check is reported explicitly; no certified result is printed for it."""
    saved = load_candidates(path)
    results = {}
    for d in sorted(saved) if dimensions is None else dimensions:
        started = time.monotonic()
        print(f'Starting verification d={d}', flush=True)
        try:
            row = replay_entry(saved[d], d, digits, prec)
            row['seconds'] = time.monotonic() - started
            results[d] = row

            def formatted_bound(value, upper):
                q = Fraction.from_float(value) * 10000
                k = -(-q.numerator // q.denominator) if upper else q.numerator // q.denominator
                sign = '-' if k < 0 else ''
                k = abs(k)
                return f'{sign}{k // 10000}.{k % 10000:04d}'
            root = formatted_bound(row['root_upper'], True)
            derived = formatted_bound(row['derived_lower'], False)
            normalized = formatted_bound(row['normalized_lower'], False)
            print(f"d={d}: VERIFIED; I^(1/d) <= {root}; derived >= {derived}; normalized >= {normalized}; root precision met={row['precision_achieved']}; seconds={row['seconds']:.1f}", flush=True)
        except Exception as error:
            results[d] = {'verified': False, 'error': str(error)}
            print(f'd={d}: FAILED verification: {error}', flush=True)
        Path(output).write_text(json.dumps(results, indent=2))
    return results


## 8. Upload and run

Use `SAVED_CANDIDATES.py` from the existing interval-objective bundle, unchanged.
Run this cell once. If the file is already present, it does not upload it again.


In [ ]:
if not Path(CANDIDATE_FILE).is_file():
    from google.colab import files
    files.upload()

if not Path(CANDIDATE_FILE).is_file():
    raise FileNotFoundError("Upload SAVED_CANDIDATES.py or change CANDIDATE_FILE.")


In [ ]:
REPLAY_RESULTS = verify_file(
    CANDIDATE_FILE,
    dimensions=DIMENSIONS,
    digits=DIGITS,
    prec=PREC,
    output="CERTIFIED_REPLAY.json",
)


## 9. Output and optional download

`VERIFIED` means every stored interval and tail bound passed recomputation.
Upper bounds are printed upward to four decimals; derived lower bounds downward.
The requested precision flag concerns the root-enclosure width. A candidate
need not be globally optimal, even if that flag is true.

The results are checkpointed after each dimension in `CERTIFIED_REPLAY.json`.
The following download lines are optional.


In [ ]:
# from google.colab import files
# files.download("CERTIFIED_REPLAY.json")


## Refactor validation

This readable implementation was run against **all 40 saved dimensions** using
python-flint 0.9.0 at 180 decimal digits. Every stored interval/tail certificate
passed, and every requested 1e-4 root-precision check passed.

The saved global upper bounds are exactly the same as in the original verifier.
The reported root and derived quantities were compared with the original replay.
The largest absolute difference across the root bounds, root gaps, and both
derived bounds was 0.0.

Additional checks compared derivative enclosures, curvature bounds and tails in
dimensions 4, 5 and 100. Invalid partitions and false stored interval bounds
were still rejected. The notebook's executable cells were also run from top to
bottom on dimension 6, including the optional class example.

Only readability, comments, variable names, and documentation have changed.
The candidate data, mathematical bounds, and verification interfaces are retained.
No optimization was repeated. The local notebook execution used the same pinned
FLINT version; it was not a test of the Colab browser itself.
